<a href="https://colab.research.google.com/github/sbmshukla/ML-Notebook-Quick-Saved/blob/main/MovieSentix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [ ]:
max_features = 10000 # vocabulary size
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words = 10000, )

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
# Insepect a sample review nd its label

sample_review = X_train[0]
sample_label = y_train[0]
print(f"Review: {sample_review}")
print(f"Label: {sample_label}")

In [ ]:
print(f"sample review as integer: {sample_review}")
print(f"sample label: {sample_label}")

In [ ]:
# Mapping of words index back to word

In [ ]:
word_index = imdb.get_word_index()

In [ ]:
print(word_index)

In [ ]:
reversed_word_index  = dict([(value, key) for (key, value) in word_index.items()])

In [ ]:
decoded_sentence = ' '.join([reversed_word_index.get(word - 3, '?') for word in sample_review])

In [ ]:
print(decoded_sentence)

In [ ]:
max_len = 500

X_train = sequence.pad_sequences(X_train, maxlen = max_len) #will apply pre padding
X_test = sequence.pad_sequences(X_test, maxlen = max_len)

In [ ]:
X_train[0]

In [ ]:
## Train Simple RNN Model

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=max_features, output_dim=128)) ## Embeding Layers
model.add(SimpleRNN(128, activation='tanh'))
model.add(Dense(1, activation = 'sigmoid'))

In [ ]:
from tensorflow.keras.optimizers import Adam

In [ ]:
# Compile
model.compile(optimizer=Adam(learning_rate=0.001),  # default lr is 0.001,
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
## Create an instance of Early Stopping Callback

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [ ]:
from tensorflow.keras.callbacks import TensorBoard
import datetime

# Create a log directory with timestamp
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,       # log weights histograms every epoch
    write_graph=True,       # visualize model graph
    write_images=False       # store model weights as images
)

In [ ]:
# Train The Model
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, callbacks=[early_stopping, tensorboard_callback])

In [ ]:
model.summary()

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Convert probabilities to 0 or 1
y_pred_labels = (y_pred >= 0.5).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred_labels)
print("Test Accuracy:", accuracy)
print(classification_report(y_test, y_pred_labels))
print(confusion_matrix(y_test, y_pred_labels))

In [ ]:
model.save('simple_rnn_imdb.h5')

In [ ]:
## Letter On We Will Play With Parameter And Improve Model Accuracy

In [ ]:
## Prediction Pipeline

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [ ]:
word_index = imdb.get_word_index()
reversed_word_index = {value:key for key, value in word_index.items()}

In [ ]:
model = load_model('simple_rnn_imdb.h5')

In [ ]:
model.summary()

In [ ]:
#Step-2 Helper Functions

In [ ]:
def decode_review(encoded_review):
  return ' '.join([reversed_word_index.get(i -3) for i in encoded_review])

In [ ]:
def preprocess_text(text, maxlen=500):
    words = text.lower().split()
    if not words:
        return np.zeros((1, maxlen), dtype=np.int32)
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=maxlen)
    return np.array(padded_review, dtype=np.int32)

In [ ]:
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input, verbose=0)
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    return sentiment, float(prediction[0][0])

In [ ]:
text = "This movie exceeded my expectations. The acting was great, the plot was engaging, and the cinematography was beautiful. I really enjoyed it."
sentiment, prob = predict_sentiment(text)
print(sentiment, prob)

In [ ]:
# We Have To Change Preprocessing And Tune Model